# RRT* — Sampling-Based Optimal Motion Planning

**CS5100 Foundations of Artificial Intelligence · Northeastern University · Spring 2026**
**Shreesh Kulkarni**

---

Reproduction of core experimental results from:
> Karaman, S. & Frazzoli, E. (2011). *Sampling-based algorithms for optimal motion planning.*
> International Journal of Robotics Research, 30(7), 846–894.

---

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | The 2D motion planning problem and three test environments |
| 2 | **RRT** — the fast but suboptimal baseline |
| 3 | **RRT\*** — two additions that achieve asymptotic optimality |
| 4 | Side-by-side comparison on all three environments |
| 5 | Quantitative analysis: path cost vs. number of iterations |

**Run all cells top-to-bottom** (Runtime → Run all, or Shift+Enter cell by cell).

In [ ]:
# Install dependencies (pre-installed on Colab; safe to run anyway)
!pip install numpy scipy matplotlib pandas --quiet

import math
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from scipy.spatial import KDTree
from collections import deque

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

print("Setup complete.")

---
## 1. The Motion Planning Problem

**Motion planning** asks: given a start and a goal in a space cluttered with obstacles,
find a collision-free path.

In 2D every configuration is a point `(x, y)`. The configuration space splits into:

- **Obstacle space** (C_obs) — positions that collide with an obstacle
- **Free space** (C_free) — all valid positions

The planner's task: connect `start → goal` entirely through C_free.

We'll test on a **10 × 10 map** with three preset obstacle configurations of
increasing difficulty. The start is always in the bottom-left region and the
goal in the top-right.

In [ ]:
class Environment:
    """2D rectangular map with axis-aligned rectangular obstacles."""

    def __init__(self, width, height, obstacles, start, goal, goal_radius=0.5):
        self.width = width
        self.height = height
        self.obstacles = obstacles   # list of (x, y, w, h)
        self.start = start
        self.goal = goal
        self.goal_radius = goal_radius

    # ---- Sampling --------------------------------------------------------

    def sample_free(self, max_attempts=10000):
        """Rejection-sample a uniform random point in free space."""
        for _ in range(max_attempts):
            x = np.random.uniform(0, self.width)
            y = np.random.uniform(0, self.height)
            if not self._point_in_any_obstacle(x, y):
                return (x, y)
        raise RuntimeError("sample_free(): map may be fully blocked")

    def _point_in_obstacle(self, x, y, ox, oy, ow, oh):
        return ox <= x <= ox + ow and oy <= y <= oy + oh

    def _point_in_any_obstacle(self, x, y):
        return any(self._point_in_obstacle(x, y, ox, oy, ow, oh)
                   for (ox, oy, ow, oh) in self.obstacles)

    # ---- Collision checking (slab method for AABBs) ----------------------

    def is_collision_free(self, p1, p2):
        """Return True if the segment p1→p2 does not intersect any obstacle."""
        x1, y1 = (p1.x, p1.y) if hasattr(p1, 'x') else p1
        x2, y2 = (p2.x, p2.y) if hasattr(p2, 'x') else p2
        if self._point_in_any_obstacle(x1, y1) or self._point_in_any_obstacle(x2, y2):
            return False
        dx, dy = x2 - x1, y2 - y1
        for (ox, oy, ow, oh) in self.obstacles:
            if self._segment_intersects_rect(x1, y1, dx, dy, ox, oy, ow, oh):
                return False
        return True

    def _segment_intersects_rect(self, x1, y1, dx, dy, ox, oy, ow, oh):
        t_min, t_max = 0.0, 1.0
        if abs(dx) < 1e-10:
            if x1 < ox or x1 > ox + ow:
                return False
        else:
            t1, t2 = (ox - x1) / dx, (ox + ow - x1) / dx
            if t1 > t2: t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
            if t_min > t_max: return False
        if abs(dy) < 1e-10:
            if y1 < oy or y1 > oy + oh:
                return False
        else:
            t1, t2 = (oy - y1) / dy, (oy + oh - y1) / dy
            if t1 > t2: t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
            if t_min > t_max: return False
        return True

    # ---- Goal check ------------------------------------------------------

    def in_goal(self, node):
        x, y = (node.x, node.y) if hasattr(node, 'x') else node
        gx, gy = self.goal
        return math.sqrt((x - gx)**2 + (y - gy)**2) <= self.goal_radius

    def free_volume(self):
        obstacle_area = sum(w * h for (_, _, w, h) in self.obstacles)
        return max((self.width * self.height - obstacle_area) * 0.85, 1.0)

    # ---- Visualisation ---------------------------------------------------

    def plot(self, ax):
        ax.set_xlim(0, self.width)
        ax.set_ylim(0, self.height)
        ax.set_aspect('equal')
        for (ox, oy, ow, oh) in self.obstacles:
            rect = patches.Rectangle(
                (ox, oy), ow, oh,
                linewidth=1, edgecolor='black', facecolor='grey', alpha=0.7, zorder=2)
            ax.add_patch(rect)
        ax.plot(*self.start, 'go', markersize=9, zorder=5, label='Start')
        ax.plot(*self.goal,  'r*', markersize=13, zorder=5, label='Goal')
        goal_circle = plt.Circle(
            self.goal, self.goal_radius,
            color='red', fill=False, linestyle='--', linewidth=1.5, zorder=3)
        ax.add_patch(goal_circle)

print("Environment class defined.")

In [ ]:
def make_env_a():
    """Env A — single large central obstacle (Easy)."""
    return Environment(
        width=10.0, height=10.0,
        obstacles=[(3.5, 3.0, 3.0, 4.0)],
        start=(1.0, 1.0), goal=(9.0, 9.0), goal_radius=0.5
    )

def make_env_b():
    """Env B — narrow passage between two tall walls (Hard)."""
    return Environment(
        width=10.0, height=10.0,
        obstacles=[
            (4.5, 0.0, 1.0, 4.0),   # lower wall
            (4.5, 5.0, 1.0, 5.0),   # upper wall  (gap is y in [4, 5])
        ],
        start=(1.0, 5.0), goal=(9.0, 5.0), goal_radius=0.5
    )

def make_env_c():
    """Env C — six smaller obstacles forming maze-like corridors (Medium)."""
    return Environment(
        width=10.0, height=10.0,
        obstacles=[
            (1.5, 1.5, 2.5, 1.0),
            (5.0, 1.0, 2.0, 2.5),
            (2.0, 4.5, 2.5, 1.0),
            (6.0, 5.0, 2.0, 1.5),
            (1.5, 7.5, 3.0, 1.0),
            (6.5, 7.0, 2.0, 2.0),
        ],
        start=(0.5, 0.5), goal=(9.5, 9.5), goal_radius=0.5
    )

print("Environment factories defined.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    (make_env_a(), "Env A — Single Obstacle\n(Easy)"),
    (make_env_b(), "Env B — Narrow Passage\n(Hard)"),
    (make_env_c(), "Env C — Maze-like\n(Medium)"),
]

for ax, (env, title) in zip(axes, configs):
    env.plot(ax)
    ax.set_title(title, fontsize=12)
    ax.legend(loc='upper left', fontsize=8)

plt.suptitle(
    "Three 10×10 Test Environments  ·  Start (●) bottom-left, Goal (★) top-right",
    y=1.03, fontsize=13
)
plt.tight_layout()
plt.show()

---
## 2. RRT: Rapidly-Exploring Random Trees

RRT (LaValle, 1998) grows a tree from the start by randomly sampling the free
space and connecting each sample back to the nearest existing node.

### The algorithm (5 steps per iteration)

```
for i in range(n_iter):
    1. Sample    x_rand  ← random free point
                          (with probability goal_bias, sample the goal directly)
    2. Nearest   x_nearest ← closest node already in the tree
    3. Steer     x_new   ← move from x_nearest toward x_rand by step_size
    4. Collision check   — if x_nearest→x_new is free, add x_new to tree
    5. Goal?     — if x_new is within goal_radius of goal → STOP, return path
```

**Key property:** RRT stops the instant it first touches the goal.
That first path is fast to find, but rarely optimal — and RRT will never improve it.

In [ ]:
# ── Node ──────────────────────────────────────────────────────────────────

class Node:
    """A single tree node with position, parent pointer, and cumulative cost."""
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.parent = None
        self.cost   = 0.0

    def __repr__(self):
        p = f"({self.parent.x:.2f},{self.parent.y:.2f})" if self.parent else "None"
        return f"Node(x={self.x:.2f}, y={self.y:.2f}, cost={self.cost:.3f}, parent={p})"


# ── Distance ──────────────────────────────────────────────────────────────

def dist(a, b):
    """Euclidean distance between two Node objects or (x, y) tuples."""
    ax, ay = (a.x, a.y) if hasattr(a, 'x') else a
    bx, by = (b.x, b.y) if hasattr(b, 'x') else b
    return math.sqrt((ax - bx)**2 + (ay - by)**2)


# ── KD-Tree spatial index ─────────────────────────────────────────────────

class NearestNeighborIndex:
    """
    KDTree-backed spatial index. Supports:
      - nearest(x, y)          → closest Node
      - within_radius(x, y, r) → all Nodes within distance r
    """
    def __init__(self):
        self._nodes  = []
        self._coords = []
        self._kd     = None

    def insert(self, node):
        self._nodes.append(node)
        self._coords.append([node.x, node.y])
        self._kd = KDTree(self._coords)

    def nearest(self, x, y):
        _, idx = self._kd.query([x, y])
        return self._nodes[idx]

    def within_radius(self, x, y, r):
        if self._kd is None:
            return []
        return [self._nodes[i] for i in self._kd.query_ball_point([x, y], r)]

print("Node, dist, NearestNeighborIndex defined.")

In [ ]:
def steer(from_node, to_point, step_size):
    """Move from from_node toward to_point by at most step_size."""
    tx, ty = to_point
    d = dist(from_node, (tx, ty))
    if d < 1e-10:
        return Node(from_node.x, from_node.y)
    ratio = min(step_size / d, 1.0)
    nx = from_node.x + ratio * (tx - from_node.x)
    ny = from_node.y + ratio * (ty - from_node.y)
    return Node(nx, ny)


def extract_path(goal_node):
    """Walk parent pointers from goal back to root; return list of (x, y)."""
    path, node = [], goal_node
    while node is not None:
        path.append((node.x, node.y))
        node = node.parent
    return list(reversed(path))


def get_children(node, tree):
    """Return all direct children of node in tree (by identity)."""
    return [n for n in tree if n.parent is node]


def rewire_radius(n_nodes, free_vol, d=2, step_size=0.5):
    """
    Compute the RRT* neighbourhood radius (Theorem 38, Karaman & Frazzoli 2011).

        r(n) = gamma * (ln(n) / n)^(1/d)

    The radius shrinks as n grows, keeping expected neighbours at O(log n).
    Clamped to step_size so it never collapses to zero for small n.
    """
    if n_nodes < 2:
        return step_size
    unit_ball_vol = math.pi
    gamma = (2.0 * (1.0 + 1.0 / d) * (free_vol / unit_ball_vol)) ** (1.0 / d)
    gamma *= 1.1  # 10% safety margin (Theorem 38 requires exceeding the minimum)
    return max(gamma * (math.log(n_nodes) / n_nodes) ** (1.0 / d), step_size)

print("steer, extract_path, get_children, rewire_radius defined.")

In [ ]:
def run_rrt(env, n_iter=2000, step_size=0.5, goal_bias=0.05, seed=None):
    """
    Standard RRT. Stops at the first solution found.

    Returns (goal_node_or_None, tree).
    """
    if seed is not None:
        np.random.seed(seed)

    root = Node(*env.start)
    tree = [root]
    index = NearestNeighborIndex()
    index.insert(root)

    for _ in range(n_iter):
        # Sample
        x_rand = env.goal if np.random.random() < goal_bias else env.sample_free()

        # Nearest + Steer
        x_nearest = index.nearest(*x_rand)
        x_new     = steer(x_nearest, x_rand, step_size)

        # Collision check
        if not env.is_collision_free((x_nearest.x, x_nearest.y),
                                      (x_new.x,     x_new.y)):
            continue

        # Add node
        x_new.parent = x_nearest
        x_new.cost   = x_nearest.cost + dist(x_nearest, x_new)
        tree.append(x_new)
        index.insert(x_new)

        # Goal check — RRT stops here
        if env.in_goal(x_new):
            return x_new, tree

    return None, tree

print("run_rrt defined.")

In [ ]:
def plot_tree_and_path(env, tree, path, title, ax=None):
    """Draw tree edges, nodes, and the solution path on ax."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 7))

    env.plot(ax)

    # Tree edges
    for node in tree:
        if node.parent is not None:
            ax.plot([node.parent.x, node.x], [node.parent.y, node.y],
                    color='lightgrey', linewidth=0.5, zorder=1)

    # Tree nodes
    ax.scatter([n.x for n in tree], [n.y for n in tree],
               s=2, color='grey', zorder=2)

    # Solution path
    if path:
        ax.plot([p[0] for p in path], [p[1] for p in path],
                color='red', linewidth=2.5, zorder=4, label='Path')

    ax.set_title(title, fontsize=11)
    ax.legend(loc='upper left', fontsize=8)

    if standalone:
        plt.tight_layout()
        plt.show()


def plot_side_by_side(env, rrt_result, rrtstar_result, suptitle=""):
    """Two-panel figure: RRT (left) vs RRT* (right) on the same environment."""
    rrt_goal, rrt_tree = rrt_result
    rrs_goal, rrs_tree = rrtstar_result

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    rrt_path = extract_path(rrt_goal) if rrt_goal else []
    rrt_cost = rrt_goal.cost if rrt_goal else float('inf')
    plot_tree_and_path(env, rrt_tree, rrt_path,
        title=f"RRT  |  {len(rrt_tree)} nodes  |  cost = {rrt_cost:.2f}",
        ax=axes[0])

    rrs_path = extract_path(rrs_goal) if rrs_goal else []
    rrs_cost = rrs_goal.cost if rrs_goal else float('inf')
    plot_tree_and_path(env, rrs_tree, rrs_path,
        title=f"RRT*  |  {len(rrs_tree)} nodes  |  cost = {rrs_cost:.2f}",
        ax=axes[1])

    if suptitle:
        fig.suptitle(f"RRT vs RRT*  —  {suptitle}", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

    if rrt_cost < float('inf') and rrs_cost < float('inf'):
        improvement = (rrt_cost - rrs_cost) / rrt_cost * 100
        print(f"RRT  cost : {rrt_cost:.3f}")
        print(f"RRT* cost : {rrs_cost:.3f}")
        print(f"Improvement : {improvement:.1f}%")

print("Visualization helpers defined.")

In [ ]:
SEED = 42

env_a = make_env_a()
rrt_goal_a, rrt_tree_a = run_rrt(env_a, n_iter=2000, step_size=0.5, goal_bias=0.05, seed=SEED)

rrt_path_a = extract_path(rrt_goal_a) if rrt_goal_a else []
cost_str   = f"{rrt_goal_a.cost:.3f}" if rrt_goal_a else "no path found"

fig, ax = plt.subplots(figsize=(7, 7))
plot_tree_and_path(env_a, rrt_tree_a, rrt_path_a,
    title=f"RRT on Env A  |  {len(rrt_tree_a)} nodes  |  cost = {cost_str}", ax=ax)
plt.tight_layout()
plt.show()

print(f"Goal reached : {rrt_goal_a is not None}")
print(f"Path cost    : {cost_str}")
print(f"Tree size    : {len(rrt_tree_a)} nodes")
print()
print("Notice: RRT stopped the moment it touched the goal.")
print("No matter how many more iterations we run, this path will NOT improve.")

### The fundamental limitation of RRT

RRT's greedy "stop at first solution" strategy has two consequences:

1. **Path quality is a lottery.** The first path depends entirely on which random
   samples happened to fall near the goal corridor. Run the cell above with
   a different `SEED` and watch the path change dramatically.

2. **More iterations don't help.** Once RRT returns, the tree stops growing.
   There is no mechanism to discover a better path later.

In simple environments this is acceptable. But in tight spaces — like Env B's
narrow passage — RRT's first path can be dramatically longer than optimal.

**Can we do better without sacrificing too much speed?**
Yes: by making two small additions to the loop.

---
## 3. RRT*: Asymptotically Optimal Planning

RRT* (Karaman & Frazzoli, 2011) adds exactly two operations to the RRT loop.
These are enough to prove **asymptotic optimality**: as iterations → ∞,
the path cost converges to the global optimum with probability 1.

| Operation | Timing | What it does |
|-----------|--------|-------------|
| **ChooseParent** | When adding `x_new` | Search all nodes within a radius; pick the one that minimises total cost from root — not just the nearest node |
| **Rewire** | After adding `x_new` | Check if any nearby node would be cheaper *through* `x_new`; if yes, re-parent it and propagate the saving downstream |

Additionally, RRT* **never stops early**. It runs all `n_iter` iterations
and tracks the best goal node found so far — so the path improves monotonically.

### Step 1: ChooseParent

In vanilla RRT, a new node always connects to its nearest neighbour:

```
x_new.parent = x_nearest
x_new.cost   = x_nearest.cost + dist(x_nearest, x_new)
```

RRT* instead considers **every node within a rewiring radius `r`**:

```
for candidate in nodes_within_radius(x_new, r):
    if candidate.cost + dist(candidate, x_new) < best_cost:
        if collision_free(candidate, x_new):
            best_parent = candidate
            best_cost   = candidate.cost + dist(candidate, x_new)

x_new.parent = best_parent
x_new.cost   = best_cost
```

This is a local optimality check: we pick the parent that gives `x_new`
the cheapest known path from the root — at the moment of insertion.

In [ ]:
def choose_parent(x_new, near_nodes, env):
    """
    Set x_new.parent to the nearby node yielding the lowest total cost from root.
    Returns x_new (modified in place). If no valid parent exists, parent stays None.
    """
    best_parent, best_cost = None, float('inf')

    for node in near_nodes:
        candidate_cost = node.cost + dist(node, x_new)
        if (candidate_cost < best_cost and
                env.is_collision_free((node.x, node.y), (x_new.x, x_new.y))):
            best_cost   = candidate_cost
            best_parent = node

    x_new.parent = best_parent
    x_new.cost   = best_cost
    return x_new

print("choose_parent defined.")

### Step 2: Rewire

After `x_new` is added to the tree, we check whether it can act as a
**shortcut** for any of its neighbours:

```
for neighbour in nodes_within_radius(x_new, r):
    if x_new.cost + dist(x_new, neighbour) < neighbour.cost:
        if collision_free(x_new, neighbour):
            neighbour.parent = x_new
            neighbour.cost   = x_new.cost + dist(x_new, neighbour)
            propagate_cost_to_descendants(neighbour)
```

The `propagate_cost` step is critical: when a node's cost drops, every node
in its subtree must be updated too. We do this with a simple BFS from the
re-parented node downward.

In [ ]:
def propagate_cost(node, tree):
    """BFS from node downward: update cost of all descendants after a rewire."""
    queue = deque([node])
    while queue:
        current = queue.popleft()
        for child in get_children(current, tree):
            child.cost = current.cost + dist(current, child)
            queue.append(child)


def rewire(x_new, near_nodes, env, tree):
    """Re-parent nearby nodes through x_new if that reduces their cost."""
    for node in near_nodes:
        if node is x_new.parent or node is x_new:
            continue
        new_cost = x_new.cost + dist(x_new, node)
        if (new_cost < node.cost and
                env.is_collision_free((x_new.x, x_new.y), (node.x, node.y))):
            node.parent = x_new
            node.cost   = new_cost
            propagate_cost(node, tree)


def run_rrt_star(env, n_iter=2000, step_size=0.5, goal_bias=0.05, seed=None):
    """
    RRT* algorithm. Runs all n_iter iterations; never stops early.

    Returns (best_goal_node_or_None, tree).
    """
    if seed is not None:
        np.random.seed(seed)

    root = Node(*env.start)
    tree = [root]
    index = NearestNeighborIndex()
    index.insert(root)

    best_goal_node = None
    free_vol = env.free_volume()

    for _ in range(n_iter):
        # Sample
        x_rand = env.goal if np.random.random() < goal_bias else env.sample_free()

        # Nearest + Steer
        x_nearest = index.nearest(*x_rand)
        x_new     = steer(x_nearest, x_rand, step_size)

        # Collision check (nearest → new)
        if not env.is_collision_free((x_nearest.x, x_nearest.y),
                                      (x_new.x,     x_new.y)):
            continue

        # Rewiring radius (shrinks as tree grows)
        r = rewire_radius(len(tree), free_vol)

        # ── RRT* Step 1: ChooseParent ──
        near_nodes = index.within_radius(x_new.x, x_new.y, r) or [x_nearest]
        x_new = choose_parent(x_new, near_nodes, env)
        if x_new.parent is None:
            continue

        # Add to tree
        tree.append(x_new)
        index.insert(x_new)

        # ── RRT* Step 2: Rewire ──
        rewire(x_new, near_nodes, env, tree)

        # Track best goal (never stop early)
        if env.in_goal(x_new):
            if best_goal_node is None or x_new.cost < best_goal_node.cost:
                best_goal_node = x_new

    return best_goal_node, tree

print("propagate_cost, rewire, run_rrt_star defined.")

In [ ]:
print("Running RRT and RRT* on Env A (2000 iterations) ...")
env_a = make_env_a()
rrt_result_a    = run_rrt     (env_a, n_iter=2000, step_size=0.5, goal_bias=0.05, seed=SEED)
rrtstar_result_a = run_rrt_star(env_a, n_iter=2000, step_size=0.5, goal_bias=0.05, seed=SEED)
plot_side_by_side(env_a, rrt_result_a, rrtstar_result_a, suptitle="Env A — Single Obstacle")

---
## 4. Comparison Across All Three Environments

Now let's stress-test both algorithms.

**Env B — Narrow Passage** is the most revealing case. The gap between the two
walls is only ~1 unit wide. RRT's random walk will find some path through it,
but that path will likely zig-zag badly. RRT* continuously rewires the tree
to straighten the route.

**Env C — Maze-like** has six smaller obstacles that force the path to navigate
multiple bends. More rewiring opportunities → larger improvement from RRT*.

In [ ]:
print("Running RRT and RRT* on Env B — Narrow Passage (3000 iterations) ...")
env_b = make_env_b()
rrt_result_b    = run_rrt     (env_b, n_iter=3000, step_size=0.5, goal_bias=0.05, seed=SEED)
rrtstar_result_b = run_rrt_star(env_b, n_iter=3000, step_size=0.5, goal_bias=0.05, seed=SEED)
plot_side_by_side(env_b, rrt_result_b, rrtstar_result_b, suptitle="Env B — Narrow Passage")

In [ ]:
print("Running RRT and RRT* on Env C — Maze-like (3000 iterations) ...")
env_c = make_env_c()
rrt_result_c    = run_rrt     (env_c, n_iter=3000, step_size=0.5, goal_bias=0.05, seed=SEED)
rrtstar_result_c = run_rrt_star(env_c, n_iter=3000, step_size=0.5, goal_bias=0.05, seed=SEED)
plot_side_by_side(env_c, rrt_result_c, rrtstar_result_c, suptitle="Env C — Maze-like")

---
## 5. Quantitative Analysis: Cost vs. Iterations

The visual comparison is compelling, but we need numbers to confirm the story.

**Experiment design:**
- Run each algorithm for N ∈ {500, 1000, 2000, 3000} iterations
- Repeat each (algorithm, N, environment) combination across **10 independent trials**
- Plot **mean ± std** of path cost for each N

**Expected behaviour:**
- **RRT**: cost is flat — the first solution never improves, so mean cost stays
  roughly constant regardless of N (with some trial-to-trial variance)
- **RRT\***: cost decreases monotonically with N — the longer it runs, the better
  it gets

> ⏱ This cell runs ~3–5 minutes on Colab. Get a coffee.

In [ ]:
def run_trial(algo_fn, env, n_iter, step_size=0.5, goal_bias=0.05, seed=0):
    """Run one trial; return cost, wall-time, success, node count."""
    t0 = time.perf_counter()
    goal_node, tree = algo_fn(env, n_iter=n_iter, step_size=step_size,
                               goal_bias=goal_bias, seed=seed)
    elapsed = time.perf_counter() - t0
    success = goal_node is not None
    return {
        'cost':    goal_node.cost if success else float('inf'),
        'time_s':  elapsed,
        'success': success,
        'n_nodes': len(tree),
    }


def run_experiment(env_name, env_fn, algo_fn, n_vals, n_trials):
    """Sweep n_vals × n_trials; return a DataFrame of results."""
    algo_name = algo_fn.__name__
    rows = []
    for n in n_vals:
        print(f"  {env_name} | {algo_name} | N={n:4d} | {n_trials} trials ...",
              end=' ', flush=True)
        t_start = time.perf_counter()
        for trial in range(n_trials):
            r = run_trial(algo_fn, env_fn(), n_iter=n, seed=trial)
            rows.append({'env': env_name, 'algo': algo_name,
                         'n_iter': n, 'trial': trial, **r})
        print(f"done ({time.perf_counter()-t_start:.1f}s)")
    return pd.DataFrame(rows)


def build_cost_dict(df, env_name, n_vals):
    """Build {rrt: {...}, rrt_star: {...}} expected by plot_cost_curves."""
    result = {}
    for algo_col, key in [('run_rrt', 'rrt'), ('run_rrt_star', 'rrt_star')]:
        sub = df[(df['env'] == env_name) & (df['algo'] == algo_col)]
        means, stds = [], []
        for n in n_vals:
            costs = sub[sub['n_iter'] == n]['cost']
            finite = costs[np.isfinite(costs)]
            means.append(finite.mean() if len(finite) else float('nan'))
            stds.append(finite.std()   if len(finite) else 0.0)
        result[key] = {'n_vals': n_vals, 'mean_costs': means, 'std_costs': stds}
    return result


def plot_cost_curves(results_dict, title):
    """Plot mean ± std path cost vs N for both algorithms."""
    fig, ax = plt.subplots(figsize=(8, 5))
    styles = {
        'rrt':      {'color': 'steelblue',  'label': 'RRT',  'ls': '--'},
        'rrt_star': {'color': 'darkorange', 'label': 'RRT*', 'ls': '-'},
    }
    for algo, data in results_dict.items():
        s = styles[algo]
        n_vals = data['n_vals']
        means  = np.array(data['mean_costs'])
        stds   = np.array(data['std_costs'])
        ax.plot(n_vals, means, color=s['color'], linestyle=s['ls'],
                linewidth=2, marker='o', markersize=5, label=s['label'])
        ax.fill_between(n_vals, means - stds, means + stds,
                        color=s['color'], alpha=0.15)
    ax.set_xlabel('Iterations (N)', fontsize=12)
    ax.set_ylabel('Path cost (Euclidean length)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("Experiment infrastructure defined.")

In [ ]:
N_VALS  = [500, 1000, 2000, 3000]
N_TRIALS = 10   # increase to 50 for paper-quality results (takes ~15 min)

env_configs = [
    ('Env A', make_env_a),
    ('Env B', make_env_b),
    ('Env C', make_env_c),
]

all_dfs = []

for env_name, env_fn in env_configs:
    print(f"\n=== {env_name} ===")
    for algo_fn in [run_rrt, run_rrt_star]:
        df = run_experiment(env_name, env_fn, algo_fn, N_VALS, N_TRIALS)
        all_dfs.append(df)

results = pd.concat(all_dfs, ignore_index=True)
print("\nAll experiments complete.")
print(results.groupby(['env', 'algo', 'n_iter'])['cost'].agg(['mean', 'std']).round(3))

In [ ]:
for env_name, _ in env_configs:
    sub = results[results['env'] == env_name]
    cost_dict = build_cost_dict(sub, env_name, N_VALS)
    plot_cost_curves(cost_dict, title=f"Cost vs Iterations — {env_name}")

---
## 6. Key Takeaways

| Observation | Explanation |
|-------------|-------------|
| **RRT cost is flat** | It stops at the first path — extra iterations explore new space but never revisit the existing path |
| **RRT* cost falls monotonically** | Every iteration is a chance to rewire the tree and find a shorter route |
| **Env B shows the biggest gap** | A narrow passage means RRT's first path must squeeze through awkwardly; RRT* smooths it out over time |
| **RRT* per-iteration overhead is mild** | The rewiring radius shrinks as `O((log n / n)^{1/2})`, keeping neighbour count at `O(log n)` — the runtime scales the same as RRT |

### Why does asymptotic optimality matter?

In safety-critical domains — robot surgery, autonomous driving, spacecraft
trajectory planning — *any* path is not enough. You need the **best** path,
or at least a guarantee that you are approaching it. RRT* provides exactly that:
a probabilistically complete, asymptotically optimal planner with negligible
overhead over its non-optimal predecessor.

### Further reading

- Karaman & Frazzoli (2011) — [Full paper](https://arxiv.org/abs/1105.1186)
- Gammell, Srinivasa & Barfoot (2014) — **Informed RRT***: biased sampling to speed up convergence
- Janson et al. (2015) — **FMT***: batch version with even faster convergence
